# 15_healthcare_analysis — 医疗板块 20% 持仓的合理性验证

> 目的: 在不改变 40% 进攻层比例的前提下, 验证 HQH 10 + XLV 10 (合计 20%) 的合理性

## 验证目标

1. **HQH/XLV 长史表现** vs VOO/QQQ (CAGR / Sharpe / Vol / Max DD)
2. **危机防御性验证** - 医疗在 2008/2020/2022 是否真的更稳?
3. **相关性分析** - 医疗与 VOO 是否真的低相关?
4. **替代配置测试** - 重新分配 20% 到其他位置是否更优?
5. **HQH vs XLV 对比** - 是否两者都需要?

## V5C 3.3a 进攻层 (40%)

```
VOO 10% / QQQ 10% / HQH 10% / XLV 10%
```

## 核心问题

- 医疗长期 alpha 弱是事实
- 但医疗作为**进攻层中的防御部分**是否合理?
- 替代方案 (e.g., 减医疗增 VOO/QQQ) 在 23.8 年回测里更优还是更差?

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tickers = ['VFINX','QQQ','HQH','XLV','IHI','VHT','VFITX','GLD','GC=F','DBC','PCRIX']
raw = yf.download(tickers, start='1999-01-01', auto_adjust=True)['Close']

print('数据起始:')
for t in tickers:
    if t in raw.columns:
        first = raw[t].first_valid_index()
        print(f'  {t:<8}: {first.date() if first else "N/A"}')

In [ ]:
def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

data = pd.DataFrame({
    'VOO': raw['VFINX'], 'QQQ': raw['QQQ'],
    'HQH': raw['HQH'], 'XLV': raw['XLV'],
    'IHI': raw['IHI'] if 'IHI' in raw.columns else None,
    'VHT': raw['VHT'] if 'VHT' in raw.columns else None,
    'VGSH': raw['VFITX'],
    'GLDM': synthesize(raw['GLD'], raw['GC=F']),
    'BCX': synthesize(raw['DBC'], raw['PCRIX']),
})
data = data.dropna(axis=1, how='all')

In [ ]:
# ============================================================
# 1. 单标的长史指标对比
# ============================================================
print('='*78)
print('1. 各资产单标的长史指标 (共同期间)')
print('='*78)

common = data[['VOO','QQQ','HQH','XLV','VGSH']].dropna()
ret = common.pct_change().dropna()
print(f'共同期间: {common.index[0].date()} → {common.index[-1].date()} ({len(common)/252:.1f} 年)')

results = []
for asset in ['VOO','QQQ','HQH','XLV','VGSH']:
    r = ret[asset]
    cum = (1+r).cumprod()
    n_y = len(r)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = r.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    corr_voo = r.corr(ret['VOO']) if asset != 'VOO' else 1.0
    results.append({'Asset':asset,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,
                    'Max DD':dd,'Calmar':cagr/abs(dd),'vs VOO':corr_voo})

df = pd.DataFrame(results).set_index('Asset')
print(df.round(3))

In [ ]:
# ============================================================
# 2. 危机期表现 (核心: 医疗是否真的防御)
# ============================================================
print('='*82)
print('2. 危机期单标的表现 - 验证医疗的防御性')
print('='*82)

crises = {
    '2008 GFC':            ('2007-10-09', '2009-03-09'),
    '2008 急跌':            ('2008-09-01', '2008-12-31'),
    '2018-Q4':              ('2018-10-01', '2018-12-31'),
    '2020 COVID 急跌':       ('2020-02-19', '2020-04-30'),
    '2020 流动性极端':        ('2020-03-09', '2020-03-23'),
    '2022 Bear (全年)':      ('2022-01-01', '2022-12-31'),
    '2022 滞胀核心':         ('2022-01-01', '2022-09-30'),
    '2025 Q1 关税':          ('2025-01-01', '2025-04-30'),
}

assets = ['VOO','QQQ','XLV','HQH']
print(f'\n{"危机":<22}', end='')
for a in assets: print(f' {a:>10}', end='')
print(f' {"XLV优势":>10}', end='')
print()
print('-'*100)
for n, (s, e) in crises.items():
    if pd.Timestamp(s) < ret.index[0]: continue
    line = f'{n:<22}'
    voo_r = (1 + ret['VOO'].loc[s:e]).prod() - 1
    for a in assets:
        r = (1 + ret[a].loc[s:e]).prod() - 1
        line += f' {r:>+9.2%}'
    xlv_r = (1 + ret['XLV'].loc[s:e]).prod() - 1
    advantage = xlv_r - voo_r
    line += f' {advantage:>+9.2%}'
    print(line)

In [ ]:
# ============================================================
# 3. 滚动 1Y 相关性 (医疗与 VOO 的关系演化)
# ============================================================
window = 252
rolling = pd.DataFrame(index=ret.index)
rolling['XLV_vs_VOO'] = ret['XLV'].rolling(window).corr(ret['VOO'])
rolling['HQH_vs_VOO'] = ret['HQH'].rolling(window).corr(ret['VOO'])
rolling['QQQ_vs_VOO'] = ret['QQQ'].rolling(window).corr(ret['VOO'])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(rolling['XLV_vs_VOO'], label='XLV vs VOO', linewidth=2, alpha=0.85)
ax.plot(rolling['HQH_vs_VOO'], label='HQH vs VOO', linewidth=2, alpha=0.85)
ax.plot(rolling['QQQ_vs_VOO'], label='QQQ vs VOO (对照)', linewidth=2, alpha=0.6, linestyle='--')
ax.axhline(0.7, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.85, color='red', linestyle=':', alpha=0.5)
ax.set_title('医疗板块 vs VOO 的 1Y 滚动相关性')
ax.set_ylabel('Correlation')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n中位相关性:')
print(f'  XLV vs VOO: {rolling["XLV_vs_VOO"].median():.3f}')
print(f'  HQH vs VOO: {rolling["HQH_vs_VOO"].median():.3f}')
print(f'  QQQ vs VOO: {rolling["QQQ_vs_VOO"].median():.3f}  (对照)')

In [ ]:
# ============================================================
# 4. 替代配置测试 - 在 40% 进攻层内调整
# ============================================================
# 完整的 V5C 3.3a 加上不同的进攻层组合

def simulate(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub = returns_df[used].dropna()
    target = np.array([target_weights[t] for t in used]); target = target/target.sum()
    cw = target.copy(); pr=[]; rd=[sub.index[0]]
    for date, dr in sub.iterrows():
        pr.append(np.sum(cw*dr.values))
        nw = cw*(1+dr.values); nw = nw/nw.sum()
        if np.max(np.abs(nw-target))*100 >= threshold_pp:
            cw = target.copy(); rd.append(date)
        else:
            cw = nw
    return pd.Series(pr, index=sub.index), rd

def metrics(rs, dates, name):
    cum = (1+rs).cumprod()
    n_y = len(rs)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = rs.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1)
    return {'Name':name,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,
            'Max DD':dd.min(),'Calmar':cagr/abs(dd.min()),'Rebal':len(dates)-1}

# 长史窗口 (含 BCX 限制)
data_long = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX']].dropna()
ret_long = data_long.pct_change().dropna()
print(f'长史窗口: {data_long.index[0].date()} → {data_long.index[-1].date()} ({len(data_long)/252:.1f} 年)')

# 6 种进攻层配置 (其余对冲层 30+10 / 防御层 20 不变)
# 注意: 因测试不含 DBMF, 用近似 V5C: GLDM 25 / BCX 15 (合并对冲) + VGSH 20
hedge = {'GLDM': 0.25, 'BCX': 0.15, 'VGSH': 0.20}

configs = {
    'A. 当前 V5C 3.3a (HQH10+XLV10)':
        {**{'VOO':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10}, **hedge},
    'B. 全去医疗 (VOO20+QQQ20)':
        {**{'VOO':0.20,'QQQ':0.20}, **hedge},
    'C. 减半医疗 (VOO15+QQQ15+XLV10)':
        {**{'VOO':0.15,'QQQ':0.15,'XLV':0.10}, **hedge},
    'D. 仅 XLV (VOO15+QQQ15+XLV10)': # 同 C
        {**{'VOO':0.15,'QQQ':0.15,'XLV':0.10}, **hedge},
    'E. 仅 HQH (VOO15+QQQ15+HQH10)':
        {**{'VOO':0.15,'QQQ':0.15,'HQH':0.10}, **hedge},
    'F. 加倍 XLV (VOO10+QQQ10+XLV20)':
        {**{'VOO':0.10,'QQQ':0.10,'XLV':0.20}, **hedge},
}

all_results = []
for name, w in configs.items():
    if name.startswith('D'): continue  # 与 C 同, skip
    r, d = simulate(ret_long, w)
    all_results.append(metrics(r, d, name))

df = pd.DataFrame(all_results).set_index('Name')
df['CAGR'] = df['CAGR'].apply(lambda x: f'{x:.2%}')
df['Vol'] = df['Vol'].apply(lambda x: f'{x:.2%}')
df['Sharpe'] = df['Sharpe'].apply(lambda x: f'{x:.3f}')
df['Max DD'] = df['Max DD'].apply(lambda x: f'{x:.2%}')
df['Calmar'] = df['Calmar'].apply(lambda x: f'{x:.3f}')
print('\n替代配置 long-history 对比 (~23.8 年):')
print(df)

In [ ]:
# ============================================================
# 5. 各配置的危机表现
# ============================================================
print('='*82)
print('5. 各配置在危机期的表现')
print('='*82)

configs_runs = {}
for name, w in configs.items():
    if name.startswith('D'): continue
    r, _ = simulate(ret_long, w)
    configs_runs[name.split('.')[0]] = r

print(f'\n{"危机":<22}', end='')
for k in configs_runs: print(f' {k:>9}', end='')
print()
print('-'*78)
for n, (s, e) in crises.items():
    if pd.Timestamp(s) < ret_long.index[0]: continue
    line = f'{n:<22}'
    for k, r in configs_runs.items():
        v = (1 + r.loc[s:e]).prod() - 1
        line += f' {v:>+8.2%}'
    print(line)

In [ ]:
# ============================================================
# 6. HQH vs XLV 直接对比
# ============================================================
print('='*70)
print('6. HQH vs XLV 直接对比 - 是否两个都需要?')
print('='*70)

# 单标的指标
for asset in ['HQH','XLV']:
    r = ret[asset]
    cum = (1+r).cumprod()
    n_y = len(r)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = r.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    print(f'\n{asset}:')
    print(f'  CAGR:        {cagr:.2%}')
    print(f'  Vol:         {vol:.2%}')
    print(f'  Sharpe:      {sharpe:.3f}')
    print(f'  Max DD:      {dd:.2%}')
    print(f'  vs VOO 相关: {r.corr(ret["VOO"]):.3f}')

# 互相相关性
corr_hqh_xlv = ret['HQH'].corr(ret['XLV'])
print(f'\nHQH vs XLV 相关性: {corr_hqh_xlv:.3f}')
if corr_hqh_xlv > 0.85:
    print('⚠️  高度相关 - 两个标的实质重叠')
elif corr_hqh_xlv > 0.7:
    print('⚠️  较高相关 - 两个标的部分重叠')
else:
    print('✓ 中等相关 - 两个标的有差异化')

In [ ]:
# ============================================================
# 7. 净值与回撤可视化
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for k, r in configs_runs.items():
    cum = (1+r).cumprod()
    axes[0].plot(cum, label=k, linewidth=1.8, alpha=0.85)
axes[0].set_title(f'各配置净值对比 ({ret_long.index[0].date()} - {ret_long.index[-1].date()}, log)', fontsize=13)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for k, r in configs_runs.items():
    cum = (1+r).cumprod()
    rm = cum.expanding().max()
    dd = (cum/rm - 1)
    axes[1].plot(dd, label=k, linewidth=1.5, alpha=0.85)
axes[1].set_title('各配置回撤对比', fontsize=13)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 解读思路

看完上面 7 项分析, 综合判断 HQH 10 + XLV 10 的合理性:

### 维度 1: 长期回报
- 医疗 CAGR 是否显著低于 VOO/QQQ? (预期: HQH 接近 VOO, XLV 略低于 VOO)
- Sharpe 是否合理? (预期: 类似 VOO)

### 维度 2: 危机防御性
- 2008/2020/2022 期间 XLV 是否真的少跌?
- HQH 是否因 CEF 杠杆反而更受伤?

### 维度 3: 与 VOO 相关性
- 中位相关性是否 < 0.85? (低于这个值才是有意义的'差异化')
- 在危机期相关性是否飙升? (大多数股票危机期相关性都飙升)

### 维度 4: 替代配置是否更优?
- 配置 B (全去医疗) Sharpe 是否高于 A?
  - 如果是: 医疗是 alpha 拖累
  - 如果否: 医疗的防御性确实有价值
- 配置 F (加倍 XLV) 是否最优?
  - 暗示是否应该减 HQH

### 维度 5: HQH 是否冗余?
- HQH vs XLV 相关性 > 0.85 → 重叠, 留一个就够
- HQH vs XLV 相关性 0.70-0.85 → 部分差异化, 两个都留有意义
- HQH vs XLV 相关性 < 0.70 → 有意义的差异化

## 决策标准

**保持 HQH 10 + XLV 10** 如果:
- 配置 A 在长史 Sharpe 不显著低于配置 B (差距 < 0.05)
- 配置 A 的 Max DD 显著优于配置 B
- HQH 与 XLV 相关性 < 0.85 (有差异化)

**调整为仅 XLV 20%** 如果:
- 配置 F 优于 A (HQH 反而拖累)
- HQH/XLV 相关性 > 0.85 (重叠)

**减少医疗** 如果:
- 配置 B (全去医疗) 显著优于 A
- 医疗的『防御性』在数据上不存在

## 心理准备 (持有 24 个月)

无论数据结论如何, 你需要在 24 个月里能够:
1. 接受医疗在牛市跑输 VOO/QQQ
2. 不在 GLP-1/AI/药价改革新闻冲击时清仓
3. 按 ±5pp 阈值机械再平衡 (即使医疗在跌)
4. 把医疗当作'进攻层中的防御', 不期望 alpha